# D2.4 · Containment at machine speed

**Function D — AI for SecOps → The Incident Responder**  ·  *Security of AI*

Builds on **[D2.3 · Scoping an agentic incident](https://spbreed.github.io/cyber-commons/lessons/D2.3.html)**.

| | |
|---|---|
| Open-source tooling | agentgateway, Keycloak |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

You have to stop it faster than it acts. That means the containment path — revoke, cut the gateway, kill the loop — is a thing built in advance, because improvising it takes longer than the incident does.

## 2 · The framework

```
   containment paths, in order of how fast they actually work

   1  revoke the credential      seconds, if it is short-lived
   2  cut it at the gateway      seconds, if there is a gateway
   3  kill the loop              minutes, if you know where it runs
   4  disable the integration    hours

   built in advance. improvised, path 1 takes longer than the incident.
```

Containment has always been a race. With an agent, the other runner got much
faster and you did not.

The numbers decide the design. An agent operating at 300 actions per minute
completes 2,400 further actions during an eight-minute approval cycle, against
about 60 under automated containment. That ratio is the argument for
pre-authorised, automated revocation of non-human identities.

The asymmetry that makes it safe: revoking a **human's** access needs care,
because a false positive locks a person out mid-shift. Revoking a **non-human**
identity is cheap to get wrong — the agent re-requests, or an on-call re-enables
it in a minute. So the two should have different policies, and almost nowhere do.

## 3 · Demo — the race, in actions rather than minutes

In [ ]:
def race(actions_per_min, human_minutes, auto_seconds=12):
    manual = actions_per_min * human_minutes
    auto   = actions_per_min * (auto_seconds/60)
    return {"manual": round(manual), "auto": round(auto),
            "ratio": round(manual/max(auto, 1e-9), 1)}

print(f"{'agent rate':>13}{'human 8min':>13}{'auto 12s':>11}{'ratio':>8}")
print("-" * 46)
for rate in (30, 120, 300, 1200):
    r = race(rate, 8)
    print(f"{rate:>9}/min{r['manual']:>13}{r['auto']:>11}{r['ratio']:>8}×")
print("\nAt 300/min an 8-minute approval costs 2,400 further actions.")

## 4 · Where it breaks — approval latency is not the only delay

In [ ]:
PATH = [
 ("detection fires",              8,   "rule evaluation + SIEM ingestion lag"),
 ("analyst picks it up",          240, "queue depth at 02:00"),
 ("analyst decides to contain",   180, "confirming it is not a false positive"),
 ("approval requested",           480, "on-call manager, out of hours"),
 ("revocation executed",          12,  "the only step anyone measures"),
]
total = sum(s for _, s, _ in PATH)
print(f"{'step':30s}{'seconds':>9}  why")
print("-" * 74)
for name, secs, why in PATH:
    print(f"{name:30s}{secs:>9}  {why}")
print(f"{'TOTAL':30s}{total:>9}  = {total/60:.0f} minutes")
print(f"\nat 300 actions/min that is {300*total/60:,.0f} further actions.")
print("The 12-second revocation is 1.3% of the elapsed time. Optimising it")
print("is not where the win is.")

## 5 · The control — pre-authorise on high-confidence signals

In [ ]:
SIGNALS = {
 "reached the cloud metadata service": 0.99,
 "read a path matching */.ssh/* or */.aws/*": 0.97,
 "egress to a host not on the allowlist": 0.90,
 "tool-call rate 20× its own baseline": 0.75,
 "activity outside usual hours": 0.30,
}
THRESHOLD = 0.70

def policy(signal, subject_is_human):
    conf = SIGNALS[signal]
    if subject_is_human:
        return f"page on-call (confidence {conf:.2f}) — human lockout needs a person"
    if conf >= THRESHOLD:
        return f"AUTO-REVOKE (confidence {conf:.2f}) — no approval in the path"
    return f"alert only (confidence {conf:.2f} < {THRESHOLD})"

for s in SIGNALS:
    print(f"{s:44s}{policy(s, False)}")
print()
print(f"{'same signal, human subject':44s}"
      f"{policy('reached the cloud metadata service', True)}")

auto_path = [("detection fires", 8), ("policy evaluates", 1), ("revocation executed", 12)]
auto_total = sum(s for _, s in auto_path)
print(f"\nautomated path: {auto_total}s vs manual {total}s "
      f"({total/auto_total:.0f}× faster)")
print(f"actions prevented at 300/min: {300*(total-auto_total)/60:,.0f}")
assert auto_total < total / 10

In [ ]:
# Verify: model the cost of getting it wrong, which is what makes it safe.
def cost_of_false_revocation(subject_is_human, agent_can_rerequest=True):
    if subject_is_human:
        return {"impact": "person locked out mid-shift", "recovery": "helpdesk, 20-60 min",
                "cost": "high"}
    if agent_can_rerequest:
        return {"impact": "task fails, agent re-requests with a reason (A2.4)",
                "recovery": "seconds to minutes", "cost": "low"}
    return {"impact": "agent stops until an on-call re-enables it",
            "recovery": "minutes", "cost": "moderate"}

for label, human in (("human subject", True), ("non-human identity", False)):
    c = cost_of_false_revocation(human)
    print(f"{label:22s}{c['cost']:10s}{c['impact']}")
print("\nThat asymmetry is the entire justification for two different policies.")

## What you just proved

The race table shows 2,400 versus 60 actions at 300/min for an eight-minute approval. The full containment path totals about 920 seconds, of which the revocation itself is 12. Four of five signals auto-revoke for non-human identities and none do for a human subject, cutting the path to 21 seconds and preventing roughly 4,500 actions.

## Your turn

Time your own containment path end to end, step by step. The revocation is almost never the slow part — queue depth and approval are, and both are policy choices rather than technical limits.

---

**Next → [D2.5 · Replay and forensics](https://spbreed.github.io/cyber-commons/lessons/D2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*